# Reproducible model script

This notebook is intended for use by our sprint 2 group. The aim is to produce a standardised script to use across laptops in order to train, test and extract evaluation metrics for the various fine tuned models. We hope that this script can be ran with a simple

```
###############
## CHANGE ME ##
###############
```
line which will determine the pretrained model which we will then fine tune.


A limitation of this procedure is that our learning regime must be fixed for each model. We conjecture that given an appropriate regime, the performance metrics should provide some semblance of an ordering on our models. We then hope to further tune the training regime of a single model. To re-emphasise, a model which performs best here will not neccersarily be the best model we could use under the perfect training regime. The models performance here we take to be indicative of the potential of the model, ie we are saying a model which performs better on this script is more promising than one which performs worse.  


The script will follow an outline proposed in notebook associated to the dataset we are looking to fine tune our models on found [HERE](https://www.kaggle.com/code/mohitagarwal17/dog-emotion-classifier-90-accuracy).

### NOTE TO SELF
Importantly, we want to use only 90% of total data in this script, we will then use held-out data in final stage of model learning regime tuning and testing etc.

We thus will need to consider this before handing out.

In [ ]:
import kagglehub

from pathlib import Path
from fastai.vision.all import *

from sklearn.model_selection import train_test_split
from collections import Counter

import torch

import matplotlib.pyplot as plt
import zipfile
from contextlib import redirect_stdout
from google.colab import files as colab_files

import shutil

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.

danielshanbalico_dog_emotion_path = kagglehub.dataset_download('danielshanbalico/dog-emotion')
print(danielshanbalico_dog_emotion_path)
mohitagarwal17_dog_img_path = kagglehub.dataset_download('mohitagarwal17/dog-img')
mohitagarwal17_dog_emotion_classifier_89_accuracy_other_default_1_path = kagglehub.model_download('mohitagarwal17/dog-emotion-classifier-89-accuracy/Other/default/1')

In [ ]:
searches = 'angry','happy','relaxed','sad'
path = Path(danielshanbalico_dog_emotion_path)

# Load the CSV containing the selected filenames
csv_url = "https://raw.githubusercontent.com/Sincanator/Group-Sprint-2/refs/heads/main/Data/emotion_classifier_data_from_kaggle/development_split.csv"
df = pd.read_csv(csv_url)

# Create files list using only filenames in CSV
files = [path / filename for filename in df["filename"]]

print(f"Number of images in subset: {len(files)}")

In [ ]:
# Create train test split
train_files, test_files = train_test_split(
    files,
    test_size=0.15,
    random_state=42,
    stratify=[f.parent.name for f in files] # <- Addition to notebook
    )

print(len(train_files))
print(len(test_files))

# Checking the class distribution in data split
print("Training:", Counter(f.parent.name for f in train_files))
print("Testing:", Counter(f.parent.name for f in test_files))

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=noop,
    splitter=RandomSplitter(valid_pct=0.2, seed=42), # generates validation set 20% of train
    get_y=parent_label, # gets labesl
    item_tfms=[Resize(400, method='pad', pad_mode='reflection')], # resizes to 400 x 400 with padding using reflection padding mode. Done since squishing features may make harder to read expression
    batch_tfms=aug_transforms(mult=1.0,
                              do_flip=True,
                              flip_vert=False,
                              max_rotate=15.0,
                              min_zoom=1.0,
                              max_zoom=1.1,
                              max_lighting=0.2,
                              max_warp=0.1) # augmentation done on the fly and in place rather than statically, changes for each training epoch to reduce overfitting
).dataloaders(train_files, bs=16) # batch size of 16 is changed from 56 since crahsed t4gpu usage limits, train data is of size 4000 * 0.85 * 0.8 = 2720 (2720 / 40 = 68)

learn = vision_learner(dls,

                       densenet121,

                       loss_func=CrossEntropyLossFlat(),
                       metrics=error_rate,
                       wd=0.1).to_fp16() # using transfer learning from pretrained resnet model resnet 101 is a large model possibly too largre. Could experiment with 34 to see if less compute gets similar results



In [ ]:
suggested_lr = learn.lr_find()
print(f"Suggested learning rate: {suggested_lr}")

In [ ]:
if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU available. Using CPU.")

learn.fine_tune(4, base_lr=4e-4) # finetune is built in to tune pretrained models, it uses adaptive lr. for 4 epochs trains class head, then whole model for 1 epoch
learn.unfreeze()
learn.fit_one_cycle(5, lr_max=slice(1e-6, 4e-4)) # three more epochs learning,
learn.recorder.plot_loss() # added to plot the loss

In [ ]:
test_dl = dls.test_dl(test_files, with_labels=True)
loss, err = learn.validate(dl=test_dl)
accuracy = (1-err)*100
print(f"Test Images: {len(test_files)}")
print(f"Correct Predictions: {round(accuracy*len(test_files)/100)}")
print(f"Incorrect Predictions: {len(test_files)-round(accuracy*len(test_files)/100)}")
print(f"Accuracy: {accuracy:.2f}%")

In [ ]:
next(learn.model.parameters()).device

save_path = Path('/content/dog_emotion_model_densenet121.pkl')

# Export the trained model
learn.cpu()
learn.export(save_path)

In [ ]:
interp = ClassificationInterpretation.from_learner(
    learn,
    dl=test_dl)
interp.plot_confusion_matrix(figsize=(4,4))

In [ ]:
interp.plot_top_losses(9, figsize=(10,7))

In [ ]:
interp.print_classification_report()

In [ ]:
print(f'results of form true | predicted')
interp.show_results(idxs=list(range(16)))

In [ ]:
# Create results folder

results_dir = Path('/content/densenet121_results')
results_dir.mkdir(exist_ok=True)

# 1. Copy exported model into results folder
shutil.copy(save_path, results_dir / save_path.name)

# 2. Save confusion matrix
interp.plot_confusion_matrix(figsize=(4, 4))
plt.savefig(results_dir / 'confusion_matrix.png',
            bbox_inches='tight', dpi=200)
plt.close('all')

# 3. Save top 9 losses
interp.plot_top_losses(9, figsize=(10, 7))
plt.savefig(results_dir / 'top_losses.png',
            bbox_inches='tight', dpi=200)
plt.close('all')

# 4. Save classification report
with open(results_dir / 'classification_report.txt', 'w') as f:
    with redirect_stdout(f):
        interp.print_classification_report()

# 5. Save 16 example predictions
interp.show_results(idxs=list(range(16)))
plt.savefig(results_dir / 'example_predictions.png',
            bbox_inches='tight', dpi=200)
plt.close('all')

# 6. Create ZIP
zip_path = Path('/content/densenet121_results.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in results_dir.iterdir():
        if file.is_file():
            zipf.write(file, arcname=file.name)

# 7. Download
colab_files.download(str(zip_path))